# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import ast
from typing import List, Dict
from datetime import datetime

import chromadb
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import AIMessage, ToolMessage, UserMessage
from lib.tooling import tool, Tool
from lib.parsers import PydanticOutputParser
from lib.vector_db import VectorStoreManager
from lib.memory import LongTermMemory, MemoryFragment

In [3]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [4]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Description about the evaluation result")

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")
judge_llm = LLM(model="gpt-4o-mini", temperature=0)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
@tool
def retrieve_game(query: str) -> List[Dict]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query], n_results=3)

    games = []
    for metadata in results["metadatas"][0]:
        games.append({
            "Platform": metadata.get("Platform"),
            "Name": metadata.get("Name"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description"),
            "Genre": metadata.get("Genre"),
            "Publisher": metadata.get("Publisher"),
        })
    return games

#### Evaluate Retrieval Tool

In [6]:
@tool
def evaluate_retrieval(question: str, retrieved_docs: List[Dict]) -> Dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Question: {question}\n\n"
        f"Retrieved documents:\n{json.dumps(retrieved_docs, indent=2)}"
    )

    response = judge_llm.invoke(prompt, response_format=EvaluationReport)
    parser = PydanticOutputParser(model_class=EvaluationReport)
    report = parser.parse(response)
    return report.model_dump()

#### Game Web Search Tool

In [7]:
@tool
def game_web_search(question: str) -> Dict:
    """
    Search the web for information about the video game industry.
    args:
    - question: a question about game industry.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    search_result = client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False,
    )

    citations = [
        {
            "title": result.get("title", "Untitled"),
            "url": result.get("url", ""),
            "snippet": result.get("content", "")[:200],
        }
        for result in search_result.get("results", [])
    ]

    return {
        "answer": search_result.get("answer", ""),
        "citations": citations,
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": question,
        },
    }

### Agent

In [8]:
INSTRUCTIONS = """
You are UdaPlay, an AI Research Agent for the video game industry.

When answering questions:
1. Use retrieve_game first to search the internal game database.
2. Use evaluate_retrieval to decide if the retrieved documents are sufficient.
3. If the documents are not useful, call game_web_search for up-to-date information.
4. Provide a clear, concise final answer based on the best available evidence.

Source and citation rules:
- If the answer comes from internal retrieval, mention it came from the internal game database and cite the game name, platform, and release year.
- If the answer uses game_web_search, you MUST include a "Sources:" section at the end of your final answer with markdown links in the format [Title](URL) for every web source used.
- Never omit citations when web search was used.

Always explain your reasoning and mention whether the answer came from internal data or the web.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.3,
)

In [9]:
def _format_tool_result(content: str) -> str:
    """Pretty-print tool output when possible."""
    try:
        parsed = json.loads(content)
        if isinstance(parsed, str):
            try:
                parsed = json.loads(parsed)
            except json.JSONDecodeError:
                parsed = ast.literal_eval(parsed)
        if isinstance(parsed, (dict, list)):
            return json.dumps(parsed, indent=2)
        return str(parsed)
    except (json.JSONDecodeError, TypeError, ValueError, SyntaxError):
        return content


def print_agent_answer(run):
    final_state = run.get_final_state()
    messages = final_state["messages"]

    print("\n--- Agent Execution Trace ---")
    for msg in messages:
        if isinstance(msg, UserMessage):
            print(f"User Query: {msg.content}")
        elif isinstance(msg, AIMessage):
            if msg.content:
                print(f"Thought/Reasoning: {msg.content}")
            if msg.tool_calls:
                for tc in msg.tool_calls:
                    print(
                        f"Tool Called: {tc.function.name} "
                        f"with args: {tc.function.arguments}"
                    )
        elif isinstance(msg, ToolMessage):
            print(f"Tool Result ({msg.name}):\n{_format_tool_result(msg.content)}")

    print("\n--- Final Response ---")
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.content:
            print(msg.content)
            break


questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
]

for question in questions:
    print("=" * 80)
    print(f"Question: {question}")
    print("-" * 80)
    run = agent.invoke(question)
    print_agent_answer(run)
    print()

Question: When was Pokémon Gold and Silver released?
--------------------------------------------------------------------------------
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


...

[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

--- Agent Execution Trace ---
User Query: When was Pokémon Gold and Silver released?
Tool Called: retrieve_game with args: {"query":"Pokémon Gold and Silver release date"}
Tool Result (retrieve_game):
[
  {
    "Platform": "Game Boy Color",
    "Name": "Pok\u00e9mon Gold and Silver",
    "YearOfRelease": 1999,
    "Description": "Second-generation Pok\u00e9mon games introducing new regions, Pok\u00e9mon, and gameplay mechanics.",
    "Genre": "Role-playing",
    "Publisher": "Nintendo"
  },
  {
    "Platform": "Game Boy Advance",
    "Name": "Pok\u00e9mon Ruby and Sapphire",
    "YearOfRelease": 2002,
    "Description": "Third-generation Pok\u00e9mon games set in the Hoenn region, featuring new Pok\u00e9mon and double battles.",
    "Genre": "Role-playing",
    "Publisher": "Nintendo"
  },
  {
    "Platform": "Nintendo Switch",
    "Name": "Mario Kart 8 Deluxe",
    "YearOfRelease": 2017,
    "Des

### (Optional) Advanced

In [10]:
def build_memory_registration_tool(ltm: LongTermMemory, owner: str, namespace: str) -> Tool:
    def _register(content: str) -> str:
        ltm.register(
            MemoryFragment(
                content=content,
                owner=owner,
                namespace=namespace,
            )
        )
        return "Saved new memory"

    return Tool(
        func=_register,
        name="register_memory",
        description=(
            "Register a new memory or preference about the user so it can be useful later as context.\n"
            "Args:\n"
            "    content: The information to save"
        ),
    )


def build_memory_search_tool(ltm: LongTermMemory, owner: str, namespace: str) -> Tool:
    def _search(content: str) -> str:
        result = ltm.search(
            query_text=content,
            owner=owner,
            namespace=namespace,
            limit=3,
        )
        return str([(fragment.content, distance) for fragment, distance in zip(
            result.fragments,
            result.metadata.get("distances", []),
        )])

    return Tool(
        func=_search,
        name="search_memory",
        description=(
            "Search for a stored memory or preference about the user.\n"
            "Args:\n"
            "    content: The information to look for"
        ),
    )


memory_db = VectorStoreManager(OPENAI_API_KEY)
ltm = LongTermMemory(memory_db)

memory_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
        "You are UdaPlay with long-term memory. Follow the same retrieval workflow as before, "
        "but also search memory for user preferences and register useful facts for future sessions."
    ),
    tools=[
        retrieve_game,
        evaluate_retrieval,
        game_web_search,
        build_memory_registration_tool(ltm, owner="user", namespace="udaplay"),
        build_memory_search_tool(ltm, owner="user", namespace="udaplay"),
    ],
    temperature=0.3,
)

# The Agent class already uses a StateMachine internally with tool and LLM steps.
print("Long-term memory agent ready.")

Long-term memory agent ready.
